# <font color="#418FDE" size="6.5" uppercase>**Optimierung verstehen**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Beschreiben eine Verlustfunktion für einen einzelnen Modellparameter. 
- Implementieren Raster- und Gradientenabstiegssuche für einfache Funktionen. 
- Analysieren Lernrate, Abbruchbedingungen, Skalierung und Verlustkurven. 


## **1. Parameter und Verlust**

### **1.1. Ein Parameter**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_01_01.jpg?v=1787637598" width="250">



>* Ein Parameter steuert die Modellvorhersage.
>* Änderungen zeigen bessere oder schlechtere Vorhersagen.

>* Parameter wirken nur im Datenkontext.
>* Gute Werte liefern realistische Vorhersagen.

>* Verlust misst Fehler für jeden Parameterwert
>* Optimierung sucht den kleinsten Verlust



In [ ]:
#@title Python-Code - Ein Parameter

# Wir untersuchen einen einzelnen Modellparameter.
# Der Verlust bewertet jede mögliche Steigung.
# Die Grafik zeigt den besten Parameterwert.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Beispieldaten zeigen Wohnfläche und Miete.
area_sqm = np.array([30, 45, 60, 75, 90], dtype=float)
rent_eur = np.array([420, 560, 690, 830, 980], dtype=float)

# Eine einfache Prüfung verhindert unpassende Datenformen.
if area_sqm.shape != rent_eur.shape:
    raise ValueError("Eingaben und Zielwerte müssen gleich lang sein.")

# Wir testen viele mögliche Werte für genau einen Parameter.
parameter_values = np.linspace(5.0, 15.0, 101)
loss_values = []

# Für jeden Parameter berechnen wir Vorhersagen und mittleren Fehler.
for parameter in parameter_values:
    predictions = parameter * area_sqm
    mean_squared_error = np.mean((predictions - rent_eur) ** 2)
    loss_values.append(mean_squared_error)

# Der kleinste Verlust markiert den besten getesteten Parameter.
loss_values = np.array(loss_values)
best_index = np.argmin(loss_values)
best_parameter = parameter_values[best_index]
best_loss = loss_values[best_index]

print(f"Getestete Parameterwerte: {len(parameter_values)}")
print(f"Bester Parameter: {best_parameter:.2f} Euro pro Quadratmeter")
print(f"Kleinster Verlust: {best_loss:.2f}")

# Die Kurve macht sichtbar, wie Verlust vom Parameter abhängt.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(parameter_values, loss_values, label="Verlustfunktion")
ax.scatter(best_parameter, best_loss, color="red", label="kleinster Verlust")

ax.set_title("Ein Parameter und seine Verlustfunktion")
ax.set_xlabel("Parameter: Euro pro Quadratmeter")
ax.set_ylabel("Mittlerer quadratischer Fehler")
ax.legend()
plt.show()



### **1.2. Verlust im Raster**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_01_02.jpg?v=1787637602" width="250">



>* Raster testet mögliche Parameterwerte systematisch
>* Verlust zeigt passende und schlechte Werte

>* Raster machen Verlustfunktionen anschaulich sichtbar
>* Mittlere Einstellungen liefern oft geringeren Verlust

>* Rasterfeinheit bestimmt Genauigkeit und Rechenaufwand
>* Verlustlandschaft zeigt Richtung besserer Parameter



In [ ]:
#@title Python-Code - Verlust im Raster

# Dieses Beispiel macht Verlustwerte im Raster sichtbar.
# Ein Parameter steuert die Steigung des Modells.
# Der kleinste Rasterverlust zeigt den besten Kandidaten.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Messwerte halten die Rechnung gut nachvollziehbar.
size_m2 = np.array([40, 55, 70, 85, 100], dtype=float)
price_k = np.array([120, 160, 210, 250, 305], dtype=float)

# Diese Prüfung schützt vor unpassenden Datenlängen.
if size_m2.shape != price_k.shape:
    raise ValueError("Größe und Preis brauchen gleich viele Werte.")

# Das Raster enthält mögliche Preissteigerungen pro Quadratmeter.
slope_grid = np.linspace(2.0, 3.6, 17)
loss_values = []

# Für jeden Rasterpunkt berechnen wir Vorhersagen und mittleren Fehler.
for slope in slope_grid:
    predictions = slope * size_m2
    errors = predictions - price_k
    mean_squared_error = np.mean(errors ** 2)
    loss_values.append(mean_squared_error)

# NumPy erleichtert das Finden des kleinsten Verlustes.
loss_values = np.array(loss_values)
best_index = int(np.argmin(loss_values))
best_slope = slope_grid[best_index]
best_loss = loss_values[best_index]

# Kurze Ausgaben verbinden Rasterwert und Verlustzahl.
print(f"Getestete Rasterpunkte: {len(slope_grid)}")
print(f"Bester Rasterwert: {best_slope:.2f} Tsd. Euro pro m²")
print(f"Kleinster mittlerer quadratischer Verlust: {best_loss:.2f}")

# Die Kurve zeigt, wie der Verlust vom Parameter abhängt.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(slope_grid, loss_values, marker="o", label="Rasterverlust")
ax.scatter(best_slope, best_loss, color="red", zorder=3, label="Bester Rasterpunkt")

# Achsenbeschriftungen machen Parameter und Verlust eindeutig.
ax.set_title("Verlustfunktion für einen einzelnen Parameter")
ax.set_xlabel("Steigung in Tsd. Euro pro m²")
ax.set_ylabel("Mittlerer quadratischer Verlust")
ax.legend()

plt.show()



### **1.3. Lokale Steigung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_01_03.jpg?v=1787637600" width="250">



>* Lokale Steigung zeigt Verluständerung nahebei
>* Sie weist die Richtung zur Verbesserung

>* Vorzeichen zeigt, ob Verlust steigt oder sinkt
>* Steilheit zeigt Empfindlichkeit kleiner Parameteränderungen

>* Lokale Steigung zeigt gezielte Verbesserungsrichtung
>* Parameteränderungen beeinflussen Vorhersagequalität messbar



In [ ]:
#@title Python-Code - Lokale Steigung

# Dieses Beispiel zeigt lokale Steigung beim Verlust.
# Ein Parameter steuert eine einfache Regressionsgerade.
# Die Grafik markiert Richtung und Stärke.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Messwerte halten die Rechnung gut nachvollziehbar.
x_values = np.array([1.0, 2.0, 3.0, 4.0])
y_values = np.array([2.0, 4.1, 5.9, 8.2])

# Diese Prüfung verhindert unpassende Datenformen.
if x_values.shape != y_values.shape:
    raise ValueError("x_values und y_values müssen gleich lang sein.")

# Der Verlust misst mittlere quadratische Vorhersagefehler.
def loss_for_slope(slope):
    predictions = slope * x_values
    errors = predictions - y_values
    return np.mean(errors ** 2)

# Eine kleine Verschiebung nähert die lokale Steigung an.
current_slope = 1.4
step_size = 0.01
loss_left = loss_for_slope(current_slope - step_size)
loss_right = loss_for_slope(current_slope + step_size)

# Die zentrale Differenz vergleicht beide Nachbarpunkte.
local_slope = (loss_right - loss_left) / (2 * step_size)
current_loss = loss_for_slope(current_slope)

# Ein negativer Wert bedeutet Verlustabnahme nach rechts.
print(f"Aktueller Parameter: {current_slope:.2f}")
print(f"Aktueller Verlust: {current_loss:.3f}")
print(f"Lokale Steigung: {local_slope:.3f}")

# Viele Parameterwerte zeigen die Verlustkurve als Landschaft.
slope_grid = np.linspace(0.5, 2.8, 120)
loss_grid = np.array([loss_for_slope(slope) for slope in slope_grid])

# Die Tangente macht die lokale Richtung sichtbar.
tangent_x = np.array([current_slope - 0.35, current_slope + 0.35])
tangent_y = current_loss + local_slope * (tangent_x - current_slope)

# Eine einzelne Achse genügt für den Vergleich.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(slope_grid, loss_grid, label="Verlustkurve")

# Punkt und Tangente markieren die unmittelbare Umgebung.
ax.scatter(current_slope, current_loss, color="red", zorder=3, label="aktueller Punkt")
ax.plot(tangent_x, tangent_y, color="red", linestyle="--", label="lokale Steigung")

# Beschriftungen erklären Parameter und Verlust.
ax.set_title("Lokale Steigung einer Verlustfunktion")
ax.set_xlabel("Parameter: Steigung der Geraden")
ax.set_ylabel("Mittlerer quadratischer Verlust")

# Die Legende verbindet Kurve, Punkt und Tangente.
ax.legend()
plt.show()



## **2. Gradientenabstieg verstehen**

### **2.1. Eindimensionale Funktion**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_02_01.jpg?v=1787637610" width="250">



>* Ein Parameter bestimmt den aktuellen Verlust
>* Lokale Steigung lenkt jeden Suchschritt

>* Steigung zeigt die nächste Suchrichtung.
>* Wiederholte Schritte senken den Verlust.

>* Glatte Kurven führen oft zuverlässig zum Minimum
>* Komplexe Kurven zeigen Grenzen und Lernwert



In [ ]:
#@title Python-Code - Eindimensionale Funktion

# Wir untersuchen Gradientenabstieg in einer Dimension.
# Die Steigung bestimmt jeden nächsten Schritt.
# Die Grafik zeigt Weg und Minimum.

import numpy as np
import matplotlib.pyplot as plt

# Diese Verlustfunktion hat ein klares Minimum.
def loss(parameter):
    return (parameter - 2.0) ** 2 + 1.0

# Die Ableitung beschreibt die lokale Steigung.
def gradient(parameter):
    return 2.0 * (parameter - 2.0)

# Diese Werte steuern den iterativen Suchprozess.
learning_rate = 0.25
start_parameter = -3.0
max_steps = 18

# Wir speichern jeden besuchten Parameterwert.
path = [start_parameter]
current_parameter = start_parameter

# Jeder Schritt geht entgegen der aktuellen Steigung.
for step in range(max_steps):
    current_gradient = gradient(current_parameter)
    current_parameter = current_parameter - learning_rate * current_gradient
    path.append(current_parameter)

# Eine einfache Prüfung schützt vor unerwarteten Formen.
path = np.array(path)
if path.size != max_steps + 1:
    raise ValueError("Die gespeicherten Schritte haben eine unerwartete Länge.")

# Für die Kurve berechnen wir viele Parameterwerte.
parameter_grid = np.linspace(-4.0, 5.0, 300)
loss_grid = loss(parameter_grid)
path_losses = loss(path)

print("Startparameter:", round(float(path[0]), 3))
print("Endparameter:", round(float(path[-1]), 3))
print("Endverlust:", round(float(path_losses[-1]), 3))
print("Erste fünf Parameter:", np.round(path[:5], 3).tolist())

# Die Grafik verbindet Verlustkurve und besuchte Schritte.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(parameter_grid, loss_grid, label="Verlustfunktion")
ax.scatter(path, path_losses, color="red", label="Gradientenschritte")

# Linien machen die Reihenfolge der Schritte sichtbar.
ax.plot(path, path_losses, color="red", alpha=0.5)
ax.set_title("Gradientenabstieg auf einer eindimensionalen Funktion")
ax.set_xlabel("Parameterwert")

ax.set_ylabel("Verlust")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **2.2. Lernrate wählen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_02_02.jpg?v=1787637612" width="250">



>* Lernrate steuert die Schrittgröße beim Abstieg
>* Steigung bestimmt Richtung, Lernrate die Stärke

>* Zu klein: stabil, aber sehr langsam
>* Zu groß: schwankend statt zielgerichtet

>* Lernrate balanciert Tempo und Stabilität.
>* Daten und Verlustkurve bestimmen passende Schritte.



In [ ]:
#@title Python-Code - Lernrate wählen

# Wir vergleichen Lernraten beim Gradientenabstieg.
# Eine einfache Verlustfunktion macht Schritte sichtbar.
# Die Grafik zeigt stabile und instabile Verläufe.

import numpy as np
import matplotlib.pyplot as plt

# Diese Funktion hat ihr Minimum bei x gleich drei.
def loss(x):
    return (x - 3.0) ** 2

# Die Ableitung zeigt die lokale Steigung der Funktion.
def gradient(x):
    return 2.0 * (x - 3.0)

# Drei Lernraten starten am gleichen Punkt.
learning_rates = [0.05, 0.4, 1.05]
start_x = -4.0
steps = 18

# Wir speichern die Verlustwerte für jede Lernrate.
histories = {}
for rate in learning_rates:
    x = start_x
    losses = [loss(x)]

    for step in range(steps):
        x = x - rate * gradient(x)
        losses.append(loss(x))

    histories[rate] = losses

# Kurze Zahlen helfen beim Lesen der Kurven.
print("Lernrate | Startverlust | Endverlust")
for rate in learning_rates:
    start_loss = histories[rate][0]
    end_loss = histories[rate][-1]
    print(f"{rate:7.2f} | {start_loss:11.2f} | {end_loss:9.2f}")

# Eine Achse zeigt alle Verlustkurven gemeinsam.
fig, ax = plt.subplots(figsize=(8, 5))
for rate in learning_rates:
    ax.plot(histories[rate], marker="o", label=f"Lernrate {rate}")

ax.set_title("Einfluss der Lernrate auf den Verlust")
ax.set_xlabel("Iterationsschritt")
ax.set_ylabel("Verlust")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **2.3. Sinnvoll stoppen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_02_03.jpg?v=1787637614" width="250">



>* Rechtzeitig stoppen spart unnötige Rechenzeit
>* Ausreichend gute Lösungen statt Perfektion

>* Kleine Verluständerungen können Nähe zum Minimum zeigen
>* Lernrate und Skalierung immer mitprüfen

>* Maximale Schritte verhindern endlose Optimierung.
>* Gutes Stoppen spart Ressourcen und bleibt stabil.



In [ ]:
#@title Python-Code - Sinnvoll stoppen

# Dieses Beispiel zeigt sinnvolles Stoppen beim Gradientenabstieg.
# Wir beobachten Verluständerung, Parameteränderung und Schrittgrenze.
# Die Kurve markiert den automatisch gewählten Stoppunkt.

import numpy as np
import matplotlib.pyplot as plt

# Diese einfache Funktion hat ihr Minimum bei x gleich drei.
def loss_function(x):
    return (x - 3.0) ** 2 + 1.0

# Der Gradient zeigt die lokale Steigung der Funktion.
def gradient_function(x):
    return 2.0 * (x - 3.0)

learning_rate = 0.2
max_steps = 40
loss_tolerance = 0.0001
parameter_tolerance = 0.0001

x_value = -4.0
x_history = [x_value]
loss_history = [loss_function(x_value)]
stop_reason = "Maximale Schrittzahl erreicht"

# Die Schleife endet, sobald eine Abbruchbedingung erfüllt ist.
for step in range(1, max_steps + 1):
    old_x = x_value
    old_loss = loss_function(old_x)

    x_value = old_x - learning_rate * gradient_function(old_x)
    new_loss = loss_function(x_value)

    x_history.append(x_value)
    loss_history.append(new_loss)

    loss_change = abs(old_loss - new_loss)
    parameter_change = abs(old_x - x_value)

    if loss_change < loss_tolerance:
        stop_reason = "Verlust verbessert sich kaum noch"
        break

    if parameter_change < parameter_tolerance:
        stop_reason = "Parameter bewegt sich kaum noch"
        break

x_history = np.array(x_history)
loss_history = np.array(loss_history)

if len(x_history) != len(loss_history):
    raise ValueError("Die Verlaufslängen passen nicht zusammen.")

print(f"Stoppgrund: {stop_reason}")
print(f"Ausgeführte Schritte: {len(x_history) - 1}")
print(f"Letzter Parameterwert: {x_history[-1]:.4f}")
print(f"Letzter Verlustwert: {loss_history[-1]:.6f}")
print(f"Letzte Verluständerung: {abs(loss_history[-2] - loss_history[-1]):.6f}")

# Die Grafik zeigt, wo der Algorithmus gestoppt hat.
x_grid = np.linspace(-5.0, 5.0, 300)
y_grid = loss_function(x_grid)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_grid, y_grid, label="Verlustfunktion")
ax.plot(x_history, loss_history, marker="o", label="Gradientenabstieg")
ax.scatter(x_history[-1], loss_history[-1], s=90, label="Stoppunkt")
ax.set_title("Sinnvoll stoppen beim Gradientenabstieg")
ax.set_xlabel("Parameterwert x")
ax.set_ylabel("Verlust")
ax.legend()
plt.show()



## **3. Konvergenz prüfen**

### **3.1. Epochen und Verlauf**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_03_01.jpg?v=1787637604" width="250">



>* Verlustkurven zeigen Fortschritt über Epochen
>* Schwankungen oder Stillstand weisen auf Probleme

>* Lernrate steuert Tempo und Stabilität.
>* Abbruch vorsichtig bei verrauschten Verlustkurven.

>* Verlustkurven zeigen Fortschritt und mögliche Probleme
>* Konvergenz bewusst prüfen, nicht blind vertrauen



In [ ]:
#@title Python-Code - Epochen und Verlauf

# Wir verfolgen Verlustwerte über mehrere Epochen.
# Drei Lernraten zeigen unterschiedliche Konvergenzverläufe.
# Die Kurve macht stabile Optimierung sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Diese einfache Verlustfunktion hat ihr Minimum bei drei.
def loss(parameter):
    return (parameter - 3.0) ** 2

# Die Ableitung zeigt die Richtung der stärksten Änderung.
def gradient(parameter):
    return 2.0 * (parameter - 3.0)

# Wir prüfen drei Lernraten mit gleichem Startwert.
learning_rates = [0.05, 0.4, 1.05]
labels = ["zu klein", "passend", "zu groß"]

# Jede Epoche speichert den aktuellen Verlustwert.
epochs = np.arange(0, 21)
histories = []

for learning_rate in learning_rates:
    parameter = -4.0
    losses = []

    for epoch in epochs:
        losses.append(loss(parameter))
        parameter = parameter - learning_rate * gradient(parameter)

    histories.append(losses)

# Kurze Zahlen helfen beim Lesen der Kurven.
print("Startverlust:", round(histories[0][0], 2))
print("Endverlust bei kleiner Lernrate:", round(histories[0][-1], 4))
print("Endverlust bei passender Lernrate:", round(histories[1][-1], 4))
print("Endverlust bei großer Lernrate:", round(histories[2][-1], 4))

# Eine einzelne Grafik vergleicht die Verlustverläufe.
fig, ax = plt.subplots(figsize=(8, 5))

for losses, label, learning_rate in zip(histories, labels, learning_rates):
    ax.plot(epochs, losses, marker="o", label=f"{label}: lr={learning_rate}")

ax.set_title("Verlustkurven über Epochen")
ax.set_xlabel("Epoche")
ax.set_ylabel("Verlust")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **3.2. Skalierung beeinflusst Konvergenz**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_03_02.jpg?v=1787637606" width="250">



>* Skalierung macht Merkmale vergleichbar
>* Gradientenabstieg konvergiert stabiler und schneller

>* Skalierung macht Gradientenabstieg ruhiger und schneller
>* Große Zahlen sollen Merkmale nicht dominieren

>* Skalierung, Lernrate und Verlustkurve gemeinsam prüfen
>* Merkmalsgrößen vor dem Training vergleichen



In [ ]:
#@title Python-Code - Skalierung beeinflusst Konvergenz

# Dieses Beispiel zeigt Skalierung beim Gradientenabstieg.
# Unskalierte Merkmale erschweren stabile Konvergenz deutlich.
# Die Verlustkurven machen den Unterschied sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen kleine, deterministische Regressionsdaten.
rng = np.random.default_rng(42)
area = rng.uniform(40, 160, 80)
noise = rng.normal(0, 8, 80)

# Der Zielwert hängt linear von der Wohnfläche ab.
price = 50 + 3 * area + noise
x_unscaled = area.reshape(-1, 1)

# Diese Prüfung verhindert unerwartete Formfehler.
if x_unscaled.shape != (80, 1):
    raise ValueError("Die Beispieldaten haben eine unerwartete Form.")

# Skalierung bringt das Merkmal in eine günstigere Größenordnung.
x_scaled = (x_unscaled - x_unscaled.mean()) / x_unscaled.std()

# Ein Bias-Term erlaubt dem Modell einen Achsenabschnitt.
ones = np.ones((x_unscaled.shape[0], 1))
design_unscaled = np.column_stack((ones, x_unscaled[:, 0]))
design_scaled = np.column_stack((ones, x_scaled[:, 0]))

# Diese Funktion führt einfachen Gradientenabstieg aus.
def gradient_descent(design_matrix, target, learning_rate, epochs):
    weights = np.zeros(design_matrix.shape[1])
    losses = []

    for epoch in range(epochs):
        predictions = design_matrix @ weights
        errors = predictions - target
        loss = np.mean(errors ** 2)
        losses.append(loss)

        gradient = (2 / len(target)) * (design_matrix.T @ errors)
        weights = weights - learning_rate * gradient

    return np.array(losses)

# Gleiche Lernrate, aber unterschiedliche Skalierung.
epochs = 60
learning_rate = 0.01
loss_unscaled = gradient_descent(design_unscaled, price, learning_rate, epochs)
loss_scaled = gradient_descent(design_scaled, price, learning_rate, epochs)

# Wir begrenzen extreme Werte nur für eine lesbare Grafik.
plot_unscaled = np.minimum(loss_unscaled, 200000)
plot_scaled = np.minimum(loss_scaled, 200000)

print("Vergleich mit gleicher Lernrate 0.01:")
print(f"Startverlust unskaliert: {loss_unscaled[0]:.0f}")
print(f"Endverlust unskaliert: {loss_unscaled[-1]:.0f}")
print(f"Startverlust skaliert: {loss_scaled[0]:.0f}")
print(f"Endverlust skaliert: {loss_scaled[-1]:.0f}")

# Die Grafik zeigt, welche Kurve stabiler sinkt.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(plot_unscaled, label="unskaliert")
ax.plot(plot_scaled, label="skaliert")

ax.set_title("Skalierung beeinflusst die Konvergenz")
ax.set_xlabel("Epoche")
ax.set_ylabel("Mittlerer quadratischer Verlust")
ax.legend()

plt.show()



### **3.3. Optimierung praktisch üben**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_03_03.jpg?v=1787637608" width="250">



>* Optimierung als wiederholte Entscheidung verstehen
>* Verlustkurven zeigen Probleme im Lernprozess

>* Lernrate steuert Tempo und Stabilität
>* Abbruchbedingungen sparen Zeit und verhindern Fehlstopps

>* Skalierung macht Gradientenabstieg stabiler
>* Verlustkurven zeigen nötige Anpassungen



In [ ]:
#@title Python-Code - Optimierung praktisch üben

# Wir vergleichen Lernraten beim Gradientenabstieg.
# Verlustkurven zeigen stabile oder instabile Konvergenz.
# Skalierung macht den Lernprozess deutlich ruhiger.

import numpy as np
import matplotlib.pyplot as plt

# Diese kleinen Daten beschreiben Wohnfläche und Preis.
area_sqm = np.array([35, 50, 65, 80, 95, 110], dtype=float)
price_k = np.array([120, 155, 190, 230, 260, 300], dtype=float)

# Eine einfache Prüfung verhindert unpassende Datenformen.
if area_sqm.shape != price_k.shape:
    raise ValueError("Fläche und Preis brauchen gleich viele Werte.")

# Skalierung zentriert die Fläche und macht Schritte vergleichbarer.
scaled_area = (area_sqm - area_sqm.mean()) / area_sqm.std()
intercept = price_k.mean()

# Diese Funktion trainiert nur die Steigung des Modells.
def run_gradient_descent(feature, learning_rate, max_epochs=60, tolerance=0.0001):
    slope = 0.0
    losses = []
    for epoch in range(max_epochs):
        predictions = intercept + slope * feature
        errors = predictions - price_k
        loss = np.mean(errors ** 2)
        losses.append(loss)
        gradient = 2 * np.mean(errors * feature)
        slope = slope - learning_rate * gradient
        if epoch > 0 and abs(losses[-2] - losses[-1]) < tolerance:
            break
    return slope, np.array(losses)

# Drei Varianten zeigen langsames, gutes und problematisches Lernen.
small_slope, small_losses = run_gradient_descent(scaled_area, 0.01)
good_slope, good_losses = run_gradient_descent(scaled_area, 0.2)
bad_slope, bad_losses = run_gradient_descent(area_sqm, 0.0002)

print("scikit-learn wird hier nicht benötigt; wir optimieren direkt mit NumPy.")
print(f"Kleine Lernrate: {len(small_losses)} Epochen, Verlust {small_losses[-1]:.1f}.")
print(f"Passende Lernrate: {len(good_losses)} Epochen, Verlust {good_losses[-1]:.1f}.")
print(f"Unskaliert zu groß: {len(bad_losses)} Epochen, Verlust {bad_losses[-1]:.1f}.")

# Eine einzige Grafik macht die Verlustkurven vergleichbar.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(small_losses, label="skaliert, Lernrate 0.01")
ax.plot(good_losses, label="skaliert, Lernrate 0.20")
ax.plot(bad_losses, label="unskaliert, Lernrate 0.0002")

ax.set_title("Verlustkurven beim Gradientenabstieg")
ax.set_xlabel("Epoche")
ax.set_ylabel("Mittlerer quadratischer Verlust")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Optimierung verstehen**</font>


In this lecture, you learned to:
- Beschreiben eine Verlustfunktion für einen einzelnen Modellparameter. 
- Implementieren Raster- und Gradientenabstiegssuche für einfache Funktionen. 
- Analysieren Lernrate, Abbruchbedingungen, Skalierung und Verlustkurven. 

In the next Lecture (Lecture B), we will go over 'Regression mit NumPy'